In [39]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
import random
import matplotlib.pyplot as plt


class M3D_Seg(Dataset):
    """
    A Pytorch Dataset class for the M3D Segmentation dataset.
    In train mode, random crops are taken from the data samples. In test mode, the center crop is taken.
    """
    def __init__(self, data_folder_dir, crop_size = 96, test_mode=False):
        self.data_folder = data_folder_dir
        self.data_samples = os.listdir(data_folder_dir)
        #self.data = data
        self.crop_size = crop_size
        self.test_mode = test_mode

    def __len__(self):
        return len(self.data_samples)

    def __getitem__(self, index):
        # Return a single data sample
        data = np.load(os.path.join(self.data_folder, self.data_samples[index]))
        data = torch.tensor(data)[0] # turn into tensor and remove the first dimension
        # Get the shape of the data tensor
        data_shape = data.shape

        # if training mode, crop a random 96x96x96 cube from the data tensor
        if not self.test_mode:
            # Generate random coordinates for the starting point of the cube
            start_x = random.randint(0, data_shape[0] - self.crop_size)
            start_y = random.randint(0, data_shape[1] - self.crop_size)
            start_z = random.randint(0, data_shape[2] - self.crop_size)

            # Crop the cube from the data tensor
            cropped_cube = data[start_x:start_x+self.crop_size, start_y:start_y+self.crop_size, start_z:start_z+self.crop_size]
        
        # if test mode, crop the 96x96x96 cube from the center data tensor
        else:
            # Crop the cube from the center of the data tensor
            start_x = (data_shape[0] - self.crop_size) // 2
            start_y = (data_shape[1] - self.crop_size) // 2
            start_z = (data_shape[2] - self.crop_size) // 2

            # Crop the cube from the data tensor
            cropped_cube = data[start_x:start_x+self.crop_size, start_y:start_y+self.crop_size, start_z:start_z+self.crop_size]

        return cropped_cube


# Create your dataset
dataset = M3D_Seg('M3D_Seg/train', crop_size=96, test_mode=False)

# Create a DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size,  shuffle=True)

# Iterate over the dataloader
for batch in dataloader:
    # Access the batched data
    print(batch.shape)

torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32

KeyboardInterrupt: 

In [40]:
# Create your dataset
test_dataset = M3D_Seg('M3D_Seg/test', crop_size=96, test_mode=True)

# Create a DataLoader
batch_size = 32
test_dataloader = DataLoader(test_dataset, batch_size=batch_size,  shuffle=True)

# Iterate over the dataloader
for batch in test_dataloader:
    # Access the batched data
    print(batch.shape)

torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])
torch.Size([32, 96, 96, 96])


KeyboardInterrupt: 

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define Encoder and Decoder as per previous sections

class EncoderBase(nn.Module):
    def __init__(self):
        super(EncoderBase, self).__init__()

    def forward(self, x):
        raise NotImplementedError("Subclasses should implement this!")

class SmallEncoder(EncoderBase):
    def __init__(self):
        super(SmallEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv3d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.encoder(x)

class AdaptiveDecoder(nn.Module):
    def __init__(self, encoder_output_shape, input_shape):
        super(AdaptiveDecoder, self).__init__()
        self.decoder = self._build_decoder(encoder_output_shape, input_shape)

    def _build_decoder(self, encoder_output_shape, input_shape):
        layers = []
        current_shape = encoder_output_shape
        
        while current_shape[2] < input_shape[2]:
            out_channels = current_shape[1] // 2
            layers.append(nn.ConvTranspose3d(current_shape[1], out_channels, kernel_size=3, stride=2, padding=1, output_padding=1))
            layers.append(nn.ReLU())
            current_shape = (current_shape[0], out_channels, current_shape[2]*2, current_shape[3]*2, current_shape[4]*2)
        
        layers.append(nn.ConvTranspose3d(current_shape[1], 1, kernel_size=3, stride=1, padding=1))
        layers.append(nn.Sigmoid())
        
        return nn.Sequential(*layers)

    def forward(self, x):
        return self.decoder(x)

class FlexibleAutoencoder(nn.Module):
    def __init__(self, encoder, input_shape):
        super(FlexibleAutoencoder, self).__init__()
        self.encoder = encoder
        with torch.no_grad():
            dummy_input = torch.randn(*input_shape)
            encoder_output_shape = self.encoder(dummy_input).shape
        self.decoder = AdaptiveDecoder(encoder_output_shape, input_shape)

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Training example
input_shape = (8, 1, 16, 16, 16)
encoder = SmallEncoder()
autoencoder = FlexibleAutoencoder(encoder, input_shape)

criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)

dummy_input = torch.randn(*input_shape)

for epoch in range(10):  # Number of epochs
    optimizer.zero_grad()
    outputs = autoencoder(dummy_input)
    loss = criterion(outputs, dummy_input)
    loss.backward()
    optimizer.step()
    print(f'Epoch [{epoch + 1}/10], Loss: {loss.item():.4f}')


Epoch [1/10], Loss: 1.2943
Epoch [2/10], Loss: 1.2918
Epoch [3/10], Loss: 1.2890
Epoch [4/10], Loss: 1.2854
Epoch [5/10], Loss: 1.2807
Epoch [6/10], Loss: 1.2744
Epoch [7/10], Loss: 1.2660
Epoch [8/10], Loss: 1.2548
Epoch [9/10], Loss: 1.2401
Epoch [10/10], Loss: 1.2212


In [46]:
input_shape_2 = (8, 3, 16, 16, 16)
dummy_input = torch.randn(*input_shape)

for epoch in range(10):  # Number of epochs
    optimizer.zero_grad()
    outputs = autoencoder(dummy_input)
    loss = criterion(outputs, dummy_input)
    loss.backward()
    optimizer.step()
    print(f'Epoch [{epoch + 1}/10], Loss: {loss.item():.4f}')

Epoch [1/10], Loss: 1.1930
Epoch [2/10], Loss: 1.1805
Epoch [3/10], Loss: 1.1649
Epoch [4/10], Loss: 1.1464
Epoch [5/10], Loss: 1.1254
Epoch [6/10], Loss: 1.1029
Epoch [7/10], Loss: 1.0805
Epoch [8/10], Loss: 1.0596
Epoch [9/10], Loss: 1.0420
Epoch [10/10], Loss: 1.0285
